# 09 · Earth Data Hub: precipitation and soil moisture

**Objective:** Open ERA5-Land lazily, inspect variable conventions, compare selected years and build area-weighted soil-moisture summaries.

**Route:** Live remote Zarr / EDH API key · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a precipitation map, comparison table and optionally explicit decadal soil-moisture bins. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/earth-data-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Configure authenticated remote access

Obtain an EDH API key from Quota & API Keys. Configure `.netrc` / Windows `_netrc` outside the repository as described in the [account guide](../docs/setup/accounts.md). Copy the current ERA5-Land **Zarr v3** URL from its catalogue access snippet into `EDH_ERA5_URL`. Keep the URL free of credentials. The `trust_env` option lets the HTTP client use your private netrc credentials. This environment uses Zarr 3; use the separate environment for Polytope.


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
import xarray as xr
from examples.climate import coordinate_slice
url = public_url(required_env("EDH_ERA5_URL"))
ds = xr.open_dataset(url, engine="zarr", chunks={}, storage_options={"client_kwargs": {"trust_env": True}})
print({"variables": list(ds.data_vars), "sizes": dict(ds.sizes)})


## 3. Inspect the variable and select a bounded region

The expected source variables are `tp` and `swvl1`. Their metadata and temporal conventions must match the catalogue. Choose `EDH_TP_CONVENTION=accumulated_m` only for a documented daily accumulated depth in metres. This example fails on unknown conventions instead of inventing unit conversion.


In [ ]:
from examples.climate import precipitation_mm
assert "tp" in ds and "swvl1" in ds
time_name = "valid_time" if "valid_time" in ds.coords else "time"
lat_name = "latitude" if "latitude" in ds.coords else "lat"
lon_name = "longitude" if "longitude" in ds.coords else "lon"
print({"tp_metadata": ds.tp.attrs, "soil_moisture_metadata": ds.swvl1.attrs})
region = ds[["tp", "swvl1"]].sel({lat_name: coordinate_slice(ds[lat_name], 58, 62), lon_name: coordinate_slice(ds[lon_name], 22, 28)})
convention = required_env("EDH_TP_CONVENTION")
if convention != "accumulated_m":
    raise ValueError("This daily example requires metadata-confirmed accumulated metres; adapt for other products.")
sample = region.tp.sel({time_name: "2020-02-05"}).squeeze().compute()
rain_mm = xr.DataArray(precipitation_mm(sample.values, convention), coords=sample.coords, dims=sample.dims, attrs={"units": "mm/day", "conversion": "metadata-confirmed daily accumulation"})
rain_mm.plot(figsize=(7, 4))
plt.title("ERA5-Land daily precipitation — bounded Finland-area rectangle")
plt.show()


## 4. Compare years using approximate cell-area weights

A regular longitude/latitude grid has smaller cell areas at higher latitudes. Cos(latitude) weights approximate spherical grid-cell area. This rectangle includes surrounding countries/sea cells; apply actual land/site boundaries in a study. Modelled soil moisture is not a peatland water-table depth.


In [ ]:
weights = np.cos(np.deg2rad(region[lat_name]))
rows = []
for year in (1951, 2020):
    chosen = region.sel({time_name: str(year)})
    if chosen.sizes.get(time_name, 0) == 0:
        continue
    # Require all daily observations for annual totals; missing cells stay NaN.
    expected_days = 366 if pd.Timestamp(f"{year}-12-31").is_leap_year else 365
    annual_total = (chosen.tp * 1000).sum(time_name, skipna=True, min_count=expected_days)
    precip = annual_total.weighted(weights).mean((lat_name, lon_name)).compute().item()
    moisture = chosen.swvl1.mean(time_name).weighted(weights).mean((lat_name, lon_name)).compute().item()
    rows.append({"year": year, "area_mean_annual_precipitation_mm": precip, "area_mean_swvl1": moisture, "available_steps": chosen.sizes[time_name]})
comparison = pd.DataFrame(rows)
comparison


## 5. Longer-period summaries are optional

Seventy years of daily chunks can consume a substantial quota. Start with one year and inspect the request volume. Enable the bounded decadal step only after estimating the reads; the source workflow compared 1951–2020 but this guide does not silently trigger that computation.


In [ ]:
RUN_LONG_PERIOD = False
if RUN_LONG_PERIOD:
    soil = region.swvl1.sel({time_name: slice("1951-01-01", "2020-12-31")})
    annual = soil.groupby(f"{time_name}.year").mean(time_name).weighted(weights).mean((lat_name, lon_name))
    # Explicit bins: 1951–1960, ..., 2011–2020; retain completeness counts.
    decade = ((annual.year - 1951) // 10) * 10 + 1951
    decadal = annual.assign_coords(decade_start=("year", decade.data)).groupby("decade_start").mean("year").compute()
    decadal.plot(marker="o", figsize=(8, 4))
    plt.ylabel("Area-weighted SWVL1 (source units)")
    plt.show()
record_provenance("edh-era5-land.json", dataset_url=url, variable_metadata={"tp": ds.tp.attrs, "swvl1": ds.swvl1.attrs}, region_wgs84=[22, 58, 28, 62], time_comparison=[1951, 2020], weighting="cos(latitude), regular grid", tp_convention=convention, long_period_enabled=RUN_LONG_PERIOD)


## Interpret and verify

This reproduces the access/analysis pattern on a smaller rectangle. A full European analysis needs a reviewed mask, quota budget and cell areas; do not casually read decades over a continent. Verify daily sampling and all annual days before using totals. Decadal bins can be incomplete if the dataset lacks any years.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [Earth Data Hub setup and authentication](https://earthdatahub.destine.eu/getting-started)
- [Earth Data Hub catalogue](https://earthdatahub.destine.eu/)
- [ECMWF ERA5-Land units and accumulations](https://confluence.ecmwf.int/spaces/CKB/pages/140385202/ERA5-Land%2Bdata%2Bdocumentation)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
